# Mathématiques appliquées à l'Intelligence Artificielle 
## 📝 Jour 3 : Probabilités et classification
## 📚 Complément du cours 

2e année Bachelor Informatique

---

> © 2025 Mohamed EL AFRIT
> Ce contenu est distribué sous licence [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr)
> [www.mohamedelafrit.com](https://www.mohamedelafrit.com)


## 🎯 Objectifs du Jour 3
- Passer de la **régression** (prédire un nombre) à la **classification** (prédire une catégorie)
- Comprendre les **probabilités conditionnelles** sur le dataset
- Implémenter un **perceptron from scratch**
- Visualiser la **frontière de décision**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DATASET FIL ROUGE — 30 développeurs
# ============================================================
# Colonnes : experience, nb_langages, niveau_etudes,
#            taille_entreprise, remote, salaire
noms_colonnes = ["experience", "nb_langages", "niveau_etudes",
                 "taille_entreprise", "remote", "salaire"]

data = np.array([
    [ 0, 2, 3,  150,  80, 28.0],  # 0  junior
    [ 1, 1, 2,   50, 100, 26.5],  # 1  autodidacte
    [ 1, 3, 4,  800,  40, 35.0],  # 2  sortie ecole
    [ 2, 2, 3,  200,  60, 32.0],  # 3
    [ 2, 4, 4, 3000,  20, 38.0],  # 4
    [ 0, 1, 1,   30,  50, 25.0],  # 5  debutant
    [ 3, 3, 3,  100,  80, 34.5],  # 6
    [ 1, 5, 4,   15, 100, 42.0],  # 7  junior polyglotte
    [ 4, 3, 3,  500,  60, 38.5],  # 8
    [ 5, 4, 4, 1200,  40, 44.0],  # 9
    [ 5, 3, 3,  300,  80, 41.0],  # 10
    [ 6, 5, 4, 2000,  30, 48.0],  # 11
    [ 6, 4, 3,  150,  90, 43.5],  # 12
    [ 7, 5, 4,  800,  50, 50.0],  # 13
    [ 7, 3, 2,   80,  70, 39.0],  # 14 exp mais Bac+2
    [ 4, 6, 5,  400, 100, 47.0],  # 15
    [ 8, 4, 4, 1500,  40, 51.0],  # 16
    [ 5, 2, 3, 4000,  10, 40.0],  # 17
    [ 9, 5, 4,  600,  60, 52.0],  # 18
    [10, 6, 4, 2500,  30, 55.0],  # 19
    [10, 4, 3,  200,  80, 48.5],  # 20
    [11, 5, 5, 1000,  50, 58.0],  # 21
    [12, 7, 4, 3500,  20, 60.0],  # 22
    [ 9, 3, 3,   60,  90, 42.0],  # 23 senior sous-paye
    [13, 6, 4,  800,  70, 62.0],  # 24
    [11, 4, 4, 5000,  10, 54.0],  # 25
    [15, 8, 4, 1200,  50, 65.0],  # 26
    [17, 6, 5, 2000,  40, 68.0],  # 27
    [20, 7, 4,  500,  60, 72.0],  # 28
    [18, 5, 3,  100,  80, 58.0],  # 29
])

experience = data[:, 0]
nb_langages = data[:, 1]
salaire = data[:, 5]
salaire_eleve = (salaire > 45).astype(int)
n = len(data)

print(f"Dataset charge : {n} developpeurs, {data.shape[1]} colonnes")
print(f"Classe 0 (<=45k) : {np.sum(salaire_eleve==0)}, Classe 1 (>45k) : {np.sum(salaire_eleve==1)}")


---
## 1. De la régression à la classification

Au lieu de prédire un salaire exact, on prédit : **salaire > 45k€ ?** (oui/non)

> **Classification binaire** : $f : \mathbb{R}^p \to \{0, 1\}$
> - $y = 0$ : salaire ≤ 45k€
> - $y = 1$ : salaire > 45k€


In [ ]:
# Visualiser les deux classes
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if y==0 else '#2ecc71' for y in salaire_eleve]
ax.scatter(experience, salaire, c=colors, s=60, edgecolors='white', lw=0.5)
ax.axhline(45, color='gray', ls='--', alpha=0.5, label='Seuil 45k€')
ax.set_xlabel("Expérience"); ax.set_ylabel("Salaire (k€)")
ax.set_title(f"Classification binaire (15 vs 15)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## 2. Probabilités conditionnelles

> $$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$


In [ ]:
# P(salaire > 45k)
print(f"P(sal>45k) = {np.mean(salaire_eleve):.2f}")

# P(sal>45k | experience > 5)
mask = experience > 5
print(f"P(sal>45k | exp>5) = {np.mean(salaire_eleve[mask]):.2f} ({np.sum(mask)} dev)")

# P(sal>45k | experience <= 5)
print(f"P(sal>45k | exp≤5) = {np.mean(salaire_eleve[~mask]):.2f}")
print(f"\nL'expérience est très prédictive !")


---
## 3. Le perceptron

> **Modèle** : $\hat{y} = \text{signe}(\mathbf{w}^T\mathbf{x} + b)$
>
> **Mise à jour (si erreur)** : $\mathbf{w} \leftarrow \mathbf{w} + \alpha(y_i - \hat{y}_i)\mathbf{x}_i$


In [ ]:
# Perceptron from scratch
X = data[:, :2]  # experience + nb_langages
y = salaire_eleve

w = np.zeros(2); b = 0.0; alpha = 0.1
for epoch in range(100):
    errs = 0
    for i in range(n):
        z = w @ X[i] + b
        yp = 1 if z > 0 else 0
        if yp != y[i]:
            err = y[i] - yp
            w += alpha * err * X[i]
            b += alpha * err
            errs += 1
    if epoch < 5 or epoch % 20 == 19:
        acc = np.mean([(1 if (w@X[j]+b)>0 else 0)==y[j] for j in range(n)])
        print(f"Epoch {epoch+1:3d} : erreurs={errs}, accuracy={acc:.0%}")

print(f"\nPoids : w=[{w[0]:.3f}, {w[1]:.3f}], b={b:.3f}")


---
## 4. Frontière de décision


In [ ]:
# Visualiser
fig, ax = plt.subplots(figsize=(8, 6))
for c, color, label in [(0,'#e74c3c','≤ 45k'),(1,'#2ecc71','> 45k')]:
    mask = y == c
    ax.scatter(X[mask,0], X[mask,1], c=color, s=70, label=label, edgecolors='white')
if abs(w[1]) > 1e-6:
    x1 = np.linspace(-1, 21, 100)
    ax.plot(x1, -(w[0]/w[1])*x1 - b/w[1], 'k--', lw=2, label='Frontière')
ax.set_xlim(-1,21); ax.set_ylim(0,9); ax.set_xlabel("Expérience"); ax.set_ylabel("Langages")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
## 5. Limite : le problème XOR


In [ ]:
# Le perceptron échoue sur XOR
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]]); y_xor = np.array([0,1,1,0])
w_x = np.zeros(2); b_x = 0.0
errs_hist = []
for ep in range(200):
    e = 0
    for i in range(4):
        z = w_x@X_xor[i]+b_x; yp = 1 if z>0 else 0
        if yp != y_xor[i]: w_x += 0.1*(y_xor[i]-yp)*X_xor[i]; b_x += 0.1*(y_xor[i]-yp); e+=1
    errs_hist.append(e)

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,5))
for c,col in [(0,'#e74c3c'),(1,'#2ecc71')]:
    m=y_xor==c; ax1.scatter(X_xor[m,0],X_xor[m,1],c=col,s=200,edgecolors='black',lw=2,zorder=3)
ax1.set_title("XOR : impossible à séparer linéairement"); ax1.grid(True, alpha=0.3)
ax2.plot(errs_hist, 'r-'); ax2.set_title("Le perceptron oscille"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("→ Solution : réseau multicouche (Jour 4) !")


---
## ✅ Résumé du Jour 3

| Régression (J1-J2) | Classification (J3) |
|---------------------|---------------------|
| Prédire un nombre | Prédire une classe {0,1} |
| $\hat{y} = \mathbf{w}^T\mathbf{x} + b$ | $\hat{y} = \text{signe}(\mathbf{w}^T\mathbf{x} + b)$ |
| MSE | Accuracy |
| Gradient descent | Correction d'erreurs |
